# Orpheus Turkish TTS Fine-Tuning with LoRA

This notebook fine-tunes [Orpheus-3B](https://huggingface.co/unsloth/orpheus-3b-0.1-pretrained) for **Turkish text-to-speech** using LoRA and the `TransformersTrainer` SDK on **Red Hat OpenShift AI**.

## Overview

Orpheus-3B is a codec language model that generates speech as discrete SNAC audio tokens. This example:

1. **Preprocesses** Turkish audio into SNAC token sequences (rank 0, inside the TrainJob)
2. **Fine-tunes** using LoRA + DDP across multiple nodes via `TransformersTrainer`
3. **Merges** the LoRA adapter into a standalone model
4. **Generates** Turkish speech from the fine-tuned model

| Feature | Description |
| --- | --- |
| **LoRA fine-tuning** | Parameter-efficient adaptation (~5.6% trainable params) |
| **DDP distributed training** | Multi-node training via `TransformersTrainer` |
| **SNAC audio codec** | 24kHz waveform to discrete token conversion |
| **Audio-only loss** | Cross-entropy on audio tokens only; text tokens masked |

### Prerequisites

- OpenShift AI (RHOAI) 3.2+ with Kubeflow Trainer v2 enabled
- A workbench with GPU (for LoRA merge and inference)
- A shared RWX PVC named `shared` (150Gi recommended)
  - **Workbench mount**: `/opt/app-root/src/shared`

## Install the Kubeflow SDK

In [ ]:
!python3 -m pip install --force-reinstall --no-cache-dir -U \
    "kubeflow @ git+https://github.com/opendatahub-io/kubeflow-sdk.git@v0.3.0+rhaiv.2"

## Configuration

Set up API authentication, PVC paths, and training hyperparameters.

### Environment variables

- `OPENSHIFT_API_URL` -- your cluster API URL (e.g. `https://api.cluster.example.com:6443`)
- `NOTEBOOK_USER_TOKEN` -- an access token for API calls

In OpenShift AI workbenches, these are typically auto-set. If not, uncomment and populate the values in the next cell.

In [ ]:
import os

from kubernetes import client as k8s

# ============================================================================
# AUTHENTICATION
# ============================================================================
# If your workbench does not auto-populate these env vars, uncomment and fill:
#
# api_server = "https://api.your-cluster.example.com:6443"
# token = "sha256~your-token-here"

api_server = os.getenv("OPENSHIFT_API_URL")
token = os.getenv("NOTEBOOK_USER_TOKEN")

if not api_server or not token:
    raise RuntimeError(
        "OPENSHIFT_API_URL and NOTEBOOK_USER_TOKEN must be set. "
        "Either set them in your environment or uncomment the values above."
    )

configuration = k8s.Configuration()
configuration.host = api_server
configuration.verify_ssl = False
configuration.api_key = {"authorization": f"Bearer {token}"}

# ============================================================================
# NAMESPACE
# ============================================================================
NAMESPACE = os.getenv("NAMESPACE", "tf-orpheus-tts")

# ============================================================================
# PVC
# ============================================================================
PVC_NAME = "shared"
NOTEBOOK_SHARED = f"/opt/app-root/src/{PVC_NAME}"

ORPHEUS_DIR = "orpheus-tts"
CKPT_DIR = f"{ORPHEUS_DIR}/checkpoints"
HF_CACHE_DIR = f"{ORPHEUS_DIR}/hf-cache"

# ============================================================================
# MLFLOW
# ============================================================================
MLFLOW_TRACKING_URI = f"http://mlflow.{NAMESPACE}.svc.cluster.local:5000"
MLFLOW_EXPERIMENT = "orpheus-turkish-tts"

# ============================================================================
# MODEL + DATASET
# ============================================================================
BASE_MODEL = "unsloth/orpheus-3b-0.1-pretrained"
HF_DATASET = "afkfatih/turkish-tts-combined-raw"
MAX_SAMPLES = 2000  # 0 = full dataset (~81K); 2000 for a quick run
MAX_SEQ_LEN = 4096

# ============================================================================
# TRAINING HYPERPARAMETERS
# ============================================================================
NUM_NODES = 2
GPUS_PER_NODE = 1
BATCH_SIZE = 4  # gradient checkpointing enabled — fits 4 on A100-80GB
GRAD_ACCUM = 2  # effective batch = 4 * 2 * 2 nodes = 16
LEARNING_RATE = 2e-5
NUM_EPOCHS = 3
EVAL_SPLIT = 0.05
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
WARMUP_RATIO = 0.05
SAVE_STEPS = 200
LOGGING_STEPS = 10

effective_batch = BATCH_SIZE * GRAD_ACCUM * NUM_NODES * GPUS_PER_NODE
print(f"API Server: {api_server}")
print(f"Namespace: {NAMESPACE}")
print(f"PVC: {PVC_NAME} -> {NOTEBOOK_SHARED}")
print(f"MLflow: {MLFLOW_TRACKING_URI} (experiment: {MLFLOW_EXPERIMENT})")
print(f"Base model: {BASE_MODEL}")
print(f"Dataset: {HF_DATASET} (max {MAX_SAMPLES} samples)")
print(
    f"Training: {NUM_NODES} nodes x {GPUS_PER_NODE} GPU, "
    f"LoRA r={LORA_R} alpha={LORA_ALPHA}, "
    f"effective batch={effective_batch}"
)

## Define the training function

The training logic lives in `train_orpheus.py` — a self-contained script with MLflow tracking, Whisper WER/CER evaluation, and audio artifact logging. The SDK serializes the function with `cloudpickle`, so the script file does not need to exist inside the training pods.

The thin wrapper below:
1. Calls `preprocess()` on rank 0 to SNAC-encode the dataset (skips if already done)
2. Waits for preprocessing on other ranks via a sentinel file
3. Calls `train()` with parameters from environment variables

In [ ]:
import sys

sys.path.insert(0, os.getcwd())
from train_orpheus import preprocess, train


def train_func():
    """Thin wrapper: preprocess on rank 0, then train on all ranks."""
    import os
    import time
    from pathlib import Path

    rank = int(os.environ.get("RANK", 0))
    pvc = "/mnt/kubeflow-checkpoints/orpheus-tts"
    hf_cache = f"{pvc}/hf-cache"
    data_dir = f"{pvc}/preprocessed"

    max_samples = int(os.environ.get("MAX_TRAIN_SAMPLES", "0"))
    max_seq_len = int(os.environ.get("MAX_SEQ_LEN", "4096"))

    # Rank 0: SNAC-encode the dataset if not already done.
    # Force PET_NNODES=1 so preprocess() doesn't shard (only rank 0 runs it).
    if rank == 0:
        os.environ["PET_NNODES"] = "1"
        os.environ["PET_NODE_RANK"] = "0"
        preprocess(
            raw_dataset=os.environ.get("HF_DATASET", "afkfatih/turkish-tts-combined-raw"),
            base_model=os.environ.get("BASE_MODEL", "unsloth/orpheus-3b-0.1-pretrained"),
            out_dir=data_dir,
            hf_cache=hf_cache,
            max_seq_len=max_seq_len,
            max_samples=max_samples,
        )

    # File-based sync — no DDP barrier during long preprocessing
    sentinel = Path(f"{data_dir}/.done")
    while not sentinel.exists():
        print(f"Rank {rank}: waiting for preprocessing (30s)...")
        time.sleep(30)

    train(
        base_model=os.environ.get("BASE_MODEL", "unsloth/orpheus-3b-0.1-pretrained"),
        hf_dataset=os.environ.get("HF_DATASET", "afkfatih/turkish-tts-combined-raw"),
        checkpoint_dir=f"{pvc}/checkpoints",
        data_dir=data_dir,
        hf_cache=hf_cache,
        max_samples=max_samples,
        max_seq_len=max_seq_len,
        batch_size=int(os.environ.get("BATCH_SIZE", "4")),
        grad_accum=int(os.environ.get("GRAD_ACCUM", "2")),
        learning_rate=float(os.environ.get("LEARNING_RATE", "2e-5")),
        num_epochs=int(os.environ.get("NUM_EPOCHS", "3")),
        save_steps=int(os.environ.get("SAVE_STEPS", "200")),
        logging_steps=int(os.environ.get("LOGGING_STEPS", "10")),
        eval_split=float(os.environ.get("EVAL_SPLIT", "0.05")),
        warmup_ratio=float(os.environ.get("WARMUP_RATIO", "0.05")),
        lora_r=int(os.environ.get("LORA_R", "16")),
        lora_alpha=int(os.environ.get("LORA_ALPHA", "32")),
        lora_dropout=float(os.environ.get("LORA_DROPOUT", "0.05")),
    )

## Configure and submit the training job

Create a `TransformersTrainer` with periodic checkpointing (every `SAVE_STEPS`, keep 3), JIT checkpointing (saves on SIGTERM), and progression tracking (visible in the OpenShift AI Dashboard).

In [ ]:
from kubeflow.trainer.rhai import TransformersTrainer
from kubeflow.trainer.rhai.transformers import PeriodicCheckpointConfig

checkpoint_config = PeriodicCheckpointConfig(
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=3,
)

trainer = TransformersTrainer(
    func=train_func,
    num_nodes=NUM_NODES,
    resources_per_node={
        "nvidia.com/gpu": GPUS_PER_NODE,
        "cpu": "4",
        "memory": "32Gi",
    },
    env={
        "BASE_MODEL": BASE_MODEL,
        "HF_DATASET": HF_DATASET,
        "MAX_TRAIN_SAMPLES": str(MAX_SAMPLES),
        "MAX_SEQ_LEN": str(MAX_SEQ_LEN),
        "LORA_R": str(LORA_R),
        "LORA_ALPHA": str(LORA_ALPHA),
        "LORA_DROPOUT": str(LORA_DROPOUT),
        "BATCH_SIZE": str(BATCH_SIZE),
        "GRAD_ACCUM": str(GRAD_ACCUM),
        "LEARNING_RATE": str(LEARNING_RATE),
        "NUM_EPOCHS": str(NUM_EPOCHS),
        "WARMUP_RATIO": str(WARMUP_RATIO),
        "SAVE_STEPS": str(SAVE_STEPS),
        "LOGGING_STEPS": str(LOGGING_STEPS),
        "EVAL_SPLIT": str(EVAL_SPLIT),
        "MLFLOW_TRACKING_URI": MLFLOW_TRACKING_URI,
        "MLFLOW_EXPERIMENT_NAME": MLFLOW_EXPERIMENT,
        "MLFLOW_TRACKING_INSECURE_TLS": "true",
    },
    packages_to_install=[
        "peft", "snac", "soundfile", "scipy", "librosa",
        "mlflow", "matplotlib", "jiwer", "openai-whisper",
    ],
    output_dir=f"pvc://{PVC_NAME}/{CKPT_DIR}",
    periodic_checkpoint_config=checkpoint_config,
    enable_jit_checkpoint=True,
    enable_progression_tracking=True,
)

print("TransformersTrainer configured:")
print(f"  Checkpointing: every {SAVE_STEPS} steps, keep 3, JIT on SIGTERM")
print("  Progress tracking: ENABLED (visible in OpenShift AI Dashboard)")
print(f"  Output: pvc://{PVC_NAME}/{CKPT_DIR}")

In [ ]:
from kubeflow.common.types import KubernetesBackendConfig
from kubeflow.trainer import TrainerClient

api_client = k8s.ApiClient(configuration)
backend_config = KubernetesBackendConfig(
    namespace=NAMESPACE,
    client_configuration=api_client.configuration,
)
client = TrainerClient(backend_config)

runtime = client.backend.get_runtime("torch-distributed")
print(f"Using runtime: {runtime.name}")

In [ ]:
JOB_NAME = client.train(trainer=trainer, runtime=runtime)
print(f"Job submitted: {JOB_NAME}")

## Monitor the training job

`TransformersTrainer` automatically injects a `KubeflowProgressCallback` that exposes real-time training metrics (step, loss, ETA) on port 28080. The **OpenShift AI Dashboard → Training Jobs** view polls this and shows live progress.

You can also stream logs and check status from the notebook:

| Method | What you see |
| --- | --- |
| **Dashboard** | Progress bar, current step/epoch, loss, estimated time remaining |
| **`get_job_logs(follow=True)`** | Live training output (loss, eval metrics, checkpoint saves) |
| **`get_job()`** | Job status, node count, runtime info |

Training 2,000 samples for 3 epochs on 2× A100 takes roughly 1-2 hours. For 20K samples / 8 epochs, expect 6-8 hours.

In [ ]:
job = client.get_job(name=JOB_NAME)
print(f"Job: {job.name}")
print(f"Status: {job.status}")
print(f"Nodes: {job.num_nodes}")
print(f"Runtime: {job.runtime.name}")
if job.steps:
    for step in job.steps:
        print(f"  {step.name}: {step.status}")

In [ ]:
# Stream training logs (Ctrl+C to stop and continue with other cells)
for logline in client.get_job_logs(JOB_NAME, follow=True):
    print(logline, end="")

In [ ]:
import time

print("Waiting for job to complete...")
while True:
    job = client.get_job(name=JOB_NAME)
    status = job.status
    print(f"  {status}")
    if status in ("Complete", "Failed"):
        break
    time.sleep(60)

print(f"\nJob finished: {job.status}")
if job.status == "Failed":
    print("Check logs above or run: client.get_job_logs(JOB_NAME)")

## Merge LoRA adapter

After training completes, merge the LoRA adapter weights into the base model to produce a standalone model that can be used for inference without PEFT.

In [ ]:
import glob

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

hf_cache = os.path.join(NOTEBOOK_SHARED, HF_CACHE_DIR)
ckpt_base = os.path.join(NOTEBOOK_SHARED, CKPT_DIR)
final_path = os.path.join(NOTEBOOK_SHARED, ORPHEUS_DIR, "final")

final_ckpt = os.path.join(ckpt_base, "final")
if os.path.isdir(final_ckpt):
    best_ckpt = final_ckpt
else:
    checkpoints = sorted(glob.glob(os.path.join(ckpt_base, "checkpoint-*")))
    if not checkpoints:
        raise FileNotFoundError(
            f"No checkpoints found at {ckpt_base}. Ensure training completed successfully."
        )
    best_ckpt = checkpoints[-1]

print(f"Merging LoRA from: {best_ckpt}")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, cache_dir=hf_cache)
tokenizer.pad_token = tokenizer.eos_token

base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, cache_dir=hf_cache, torch_dtype=torch.bfloat16,
)

model = PeftModel.from_pretrained(base, best_ckpt)
model = model.merge_and_unload()

os.makedirs(final_path, exist_ok=True)
model.save_pretrained(final_path, safe_serialization=True)
tokenizer.save_pretrained(final_path)

print(f"Merged model saved to {final_path}")
del model, base
torch.cuda.empty_cache()

## Results

> Screenshots below are from a reference run: **20K samples, 8 epochs, 2× A100-80GB, LoRA r=32/α=64**, tracked in MLflow on OpenShift AI.

### Training & eval loss

![Training Loss](images/training_loss.png)

### In-training WER/CER (Whisper ASR)

![WER CER Progress](images/wer_cer_progress.png)

### Full MLflow dashboard

![Dashboard](images/dashboard.png)

### MLflow Traces — inference pipeline

![MLflow Traces](images/ui_traces_pipeline.png)

### MLflow Artifacts — step-indexed audio

![MLflow Artifacts](images/ui_artifacts_audio.png)

### Post-training evaluation — baseline vs fine-tuned

| Metric | Baseline | Fine-tuned | Δ |
| --- | --- | --- | --- |
| **WER mean** | 1.576 | **0.723** | −0.854 |
| **CER mean** | 1.224 | **0.410** | −0.814 |
| **eval_loss** | 9.50 | **4.35** | −5.15 |

### Per-sentence WER & CER

![WER CER Bars](images/eval_wer_cer_bars.png)

### Per-sentence CER improvement (Δ)

![CER Delta](images/eval_cer_delta.png)

Audio samples available on the [HuggingFace model card](https://huggingface.co/AbDhumal/orpheus-3b-turkish-tts-v2#audio-samples).

## Generate Turkish speech

Load the merged model and generate audio for a few Turkish sentences to verify the fine-tuning worked.

The pipeline:
1. Tokenize Turkish text with the Llama-3 tokenizer
2. Generate SNAC audio tokens with the fine-tuned model
3. Decode tokens back to a 24kHz waveform via the SNAC codec

In [ ]:
!python3 -m pip install -q snac

import IPython.display as ipd
import torch
from snac import SNAC
from transformers import AutoModelForCausalLM, AutoTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

LLAMA_VOCAB = 128_256
CODE_OFFSET = LLAMA_VOCAB + 10
SNAC_SR = 24_000
TOK_SOH = LLAMA_VOCAB + 3
TOK_EOH = LLAMA_VOCAB + 4
TOK_SOA = LLAMA_VOCAB + 5
TOK_EOA = LLAMA_VOCAB + 6
TOK_SOS = LLAMA_VOCAB + 1
TOK_EOT = LLAMA_VOCAB + 9

N_CODEBOOK = 4096
N_PER_FRAME = 7

print("Loading fine-tuned model...")
ft_tokenizer = AutoTokenizer.from_pretrained(final_path)
ft_tokenizer.pad_token = ft_tokenizer.eos_token
ft_model = (
    AutoModelForCausalLM
    .from_pretrained(final_path, torch_dtype=torch.bfloat16)
    .eval()
    .to(device)
)

print("Loading SNAC decoder...")
snac_model = SNAC.from_pretrained("hubertsiuzdak/snac_24khz").eval().to(device)


def build_prompt(text):
    ids = ft_tokenizer.encode(text, add_special_tokens=False) + [TOK_EOT]
    return [TOK_SOH] + ids + [TOK_EOH, TOK_SOA, TOK_SOS]


def snac_decode(token_ids):
    audio_ids = [t for t in token_ids if t >= CODE_OFFSET]
    n = len(audio_ids) // N_PER_FRAME
    if n == 0:
        return None
    audio_ids = audio_ids[: n * N_PER_FRAME]
    l0, l1, l2 = [], [], []
    for f in range(n):
        g = audio_ids[N_PER_FRAME * f : N_PER_FRAME * (f + 1)]
        l0.append((g[0] - CODE_OFFSET) % N_CODEBOOK)
        l1.append((g[1] - CODE_OFFSET) % N_CODEBOOK)
        l2.append((g[2] - CODE_OFFSET) % N_CODEBOOK)
        l2.append((g[3] - CODE_OFFSET) % N_CODEBOOK)
        l1.append((g[4] - CODE_OFFSET) % N_CODEBOOK)
        l2.append((g[5] - CODE_OFFSET) % N_CODEBOOK)
        l2.append((g[6] - CODE_OFFSET) % N_CODEBOOK)

    def _t(x):
        return torch.tensor(x, dtype=torch.long).unsqueeze(0).to(device)

    wav = snac_model.decode([_t(l0), _t(l1), _t(l2)])
    return wav.squeeze().cpu().float().detach().numpy()


def generate_speech(text, max_new_tokens=1500):
    prompt = build_prompt(text)
    inp = torch.tensor([prompt], dtype=torch.long, device=device)
    with torch.inference_mode():
        out = ft_model.generate(
            inp,
            max_new_tokens=max_new_tokens,
            min_new_tokens=max(80, len(text) * 7),
            do_sample=True,
            temperature=0.3,
            top_p=0.9,
            repetition_penalty=1.15,
            eos_token_id=TOK_EOA,
        )
    new_ids = out[0][len(prompt) :].cpu().tolist()
    if TOK_EOA in new_ids:
        new_ids = new_ids[: new_ids.index(TOK_EOA)]
    return snac_decode(new_ids)


# Test sentences
test_sentences = [
    ("welcome", "istanbul'a hos geldiniz."),
    ("flight", "sayin yolcularimiz, ucusumuz yaklasik iki saat surecektir."),
    ("farewell", "tesekkur ederiz, iyi yolculuklar dileriz."),
]

for label, text in test_sentences:
    print(f"\n[{label}] {text}")
    wav = generate_speech(text)
    if wav is not None:
        duration = len(wav) / SNAC_SR
        print(f"  Duration: {duration:.2f}s")
        ipd.display(ipd.Audio(wav, rate=SNAC_SR))
    else:
        print("  Failed to generate audio")

## Cleanup

Delete the training job to free cluster resources.

The model, dataset, and checkpoints remain on the PVC for future use.

In [ ]:
client.delete_job(name=JOB_NAME)
print(f"Job {JOB_NAME} deleted")

## Summary

You have fine-tuned Orpheus-3B for Turkish TTS using LoRA on Red Hat OpenShift AI.

For production-quality results, increase `MAX_SAMPLES` to 20,000+, `NUM_EPOCHS` to 8, and use `LORA_R=32` / `LORA_ALPHA=64`. MLflow tracking is enabled by default — view experiments in the OpenShift AI Dashboard.

See the other examples in `examples/trainer/` for FSDP, DeepSpeed, Kueue, and S3 checkpoint storage.